In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns, one_hot_encode
from src.pipelines import fit_and_predict, train_validate_models
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [2]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)

In [3]:
TRAFFIC_FEATURES_RAW = ["date", "sessions", "unique_visitors"]
SALE_FEATURES_RAW = ["date", "Revenue", "COGS"]

TRAFFIC_ENGINEER = ["date", "returning_rate"]

In [11]:
df = pd.DataFrame(
    {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = add_time_features(df, date_col="date")
df = df.merge(df_sales[SALE_FEATURES_RAW], on="date", how="left")
# add feature year_after_shock = max(0, year - 2019)
# add feature shock_period = 1 if date is between 2019-03-01 and 2022-12-31, else 0

df["year_after_shock"] = (df["date"].dt.year - 2019).clip(lower=0)
df["shock_period"] = ((df["date"] >= "2019-03-01") & (df["date"] <= "2022-12-31")).astype(int)

# List all feature columns
feature_columns = list_feature_columns(df)
print(feature_columns["all_features"])
df.tail()



['year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend', 'day_to_month_end', 'day_from_month_start', 'year_after_shock', 'shock_period']


,date,year,month,day,day_of_week,week_of_year,is_month_end,is_month_start,is_weekend,day_to_month_end,day_from_month_start,Revenue,COGS,year_after_shock,shock_period
4195,2024-06-27,2024,6,27,3,26,0,0,0,3,-4,NaN,NaN,5,0
4196,2024-06-28,2024,6,28,4,26,0,0,0,2,-3,NaN,NaN,5,0
4197,2024-06-29,2024,6,29,5,26,0,0,1,1,-2,NaN,NaN,5,0
4198,2024-06-30,2024,6,30,6,26,1,0,1,0,-1,NaN,NaN,5,0
4199,2024-07-01,2024,7,1,0,27,0,1,0,30,0,NaN,NaN,5,0


In [9]:
categorical_cols = ["date_of_week", "month", "day", "is_weekend",
                    "shock_period"]
df = one_hot_encode(df, categorical_cols)

In [ ]:
df.

In [10]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

result = train_validate_models(
    df=df,
    model_config=model_config,
    train_range=("2013-01-01", "2021-12-31"),
    predict_range=("2022-01-01", "2022-12-31"),
)

result["metrics"]



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001643 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 172
[LightGBM] [Info] Number of data points in the train set: 3287, number of used features: 53
[LightGBM] [Info] Start training from score 4417167.630657
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000492 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 172
[LightGBM] [Info] Number of data points in the train set: 3287, number of used features: 53
[LightGBM] [Info] Start training from score 3819765.263738


{'Revenue': {'mae': 635295.8614823294,
  'rmse': 875941.594965312,
  'mape': 21.18021664653769,
  'smape': 22.461735847451706,
  'r2': 0.7261340562103745},
 'COGS': {'mae': 522622.95062922605,
  'rmse': 689824.8493358524,
  'mape': 20.798457406681138,
  'smape': 21.29253482478842,
  'r2': 0.7763212772306499}}

In [ ]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

submission = fit_and_predict(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)
submission.head()
submission_name = "shocku"
submission.to_csv(project_root / f"submissions/{submission_name}.csv", index=False)